# Model Comparison: VMamba-Tiny vs ResNet50 vs ViT-Tiny
This notebook provides a comprehensive comparison between **MedicalVMamba (Tiny)**, **ResNet50**, and **ViT-Tiny** models on MedMNIST.

## Areas of Comparison:
1. **Training & Validation Curves**: Analyzing loss and accuracy/F1-score trajectories.
2. **Confusion Matrices & Test Loss**: Detailed evaluation of per-class predictive performance, alongside a numerical comparison of Test VS Validation Loss.
3. **Efficiency Focus**: Parameter count, inference latency, FLOPs, and asymptotic time complexity.

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import medmnist
from medmnist import INFO
from torch.utils.data import DataLoader
from torchvision import transforms
import sys
sys.path.append(r"d:\MedMamba-XAI\src")
from medical_mamba.models.medical_vmamba import build_model
from medical_mamba.models.resnet_baseline import ResNetBaseline
from medical_mamba.models.vit_baseline import ViTBaseline
from torch.nn import CrossEntropyLoss

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Loss and Validation Curves
We analyze the `metrics.csv` files generated during the training phase. The MedicalVMamba model ran for 100 epochs, while ResNet50 typically ran for ~40 epochs.

In [ ]:
datasets = ['bloodmnist', 'dermamnist', 'octmnist', 'pathmnist']

base_dir = r"d:\MedMamba-XAI\runs"
mamba_dir = os.path.join(base_dir, "medical_mamba", "single_task", "patch_16")
resnet_dir = os.path.join(base_dir, "resnet_baseline", "single_task")

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
fig.suptitle('Training Loss and Validation F1-Score Comparison', fontsize=16, y=1.02)

for i, ds in enumerate(datasets):
    try:
        # Load configs
        mamba_df = pd.read_csv(os.path.join(mamba_dir, ds, 'metrics.csv'))
        resnet_df = pd.read_csv(os.path.join(resnet_dir, ds, 'metrics.csv'))
        
        # Train Filter
        mamba_train = mamba_df[mamba_df['split'] == 'train']
        mamba_val = mamba_df[mamba_df['split'] == 'val']
        
        resnet_train = resnet_df[resnet_df['split'] == 'train']
        resnet_val = resnet_df[resnet_df['split'] == 'val']
        
        # Plot Loss
        ax_loss = axes[i, 0]
        ax_loss.plot(mamba_train['epoch'], mamba_train['total_loss'], label='VMamba-Tiny Train Loss', color='blue')
        ax_loss.plot(resnet_train['epoch'], resnet_train['total_loss'], label='ResNet50 Train Loss', color='orange')
        ax_loss.set_title(f'{ds.upper()} - Training Loss')
        ax_loss.set_xlabel('Epoch')
        ax_loss.set_ylabel('Loss')
        ax_loss.legend()

        # Plot Val F1
        ax_val = axes[i, 1]
        ax_val.plot(mamba_val['epoch'], mamba_val['avg_f1_macro'], label='VMamba-Tiny Val F1', color='green')
        ax_val.plot(resnet_val['epoch'], resnet_val['avg_f1_macro'], label='ResNet50 Val F1', color='red')
        ax_val.set_title(f'{ds.upper()} - Validation F1-Score')
        ax_val.set_xlabel('Epoch')
        ax_val.set_ylabel('F1-Score')
        ax_val.legend()
        
    except FileNotFoundError as e:
        print(f"Metrics not found for {ds}: {e}")

plt.tight_layout()
plt.show()

## 2. Confusion Matrices
*Note: Because exact trained model checkpoints were not stored locally in this workspace, we demonstrate the matrix computation by instantiating the models (either randomly initialized or with dummy weights/pretrained backbones if supported) and passing MedMNIST validation sets. If you have fine-tuned checkpoints ready, you can simply load them into `mamba_model` and `resnet_model` just below.*

In [ ]:
def get_medmnist_loader(dataset_name, split='test', batch_size=64):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info['python_class'])
    
    data_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5], std=[.5])
    ])
    
    dataset = DataClass(split=split, transform=data_transform, download=True)
    loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=False)
    return loader, info['n_channels'], len(info['label'])

def plot_confusion_matrix(y_true, y_pred, title, ax, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

# Try to load checkpoints or fallback to random
def load_weights(model, path):
    try:
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=device))
            print(f"Loaded weights from {path}")
    except Exception as e:
        print(f"Could not load {path}, using random weights. Error: {e}")

datasets_cm = ['bloodmnist', 'dermamnist', 'octmnist', 'pathmnist'] 
fig, axes = plt.subplots(len(datasets_cm), 3, figsize=(20, 6 * len(datasets_cm)))

loss_fn = CrossEntropyLoss()
loss_records = []

base_dir = r"d:\MedMamba-XAI\runs"

for i, ds in enumerate(datasets_cm):
    test_loader, n_channels, n_classes = get_medmnist_loader(ds, split='test', batch_size=64)
    task_configs = [(ds, n_classes)]
    
    # Init Mamba
    try: mamba_model = build_model(task_configs, model_size="tiny", patch_size=16, in_chans=n_channels).to(device)
    except TypeError: mamba_model = build_model(task_configs, model_size="tiny", patch_size=16).to(device)
    mamba_path = os.path.join(base_dir, "medical_mamba", "single_task", "patch_16", ds, "checkpoints", "best_model.pth")
    load_weights(mamba_model, mamba_path)
    mamba_model.eval()
    
    # Init ResNet
    resnet_model = ResNetBaseline(task_configs, pretrained=False).to(device)
    if n_channels == 1:
        resnet_model.backbone[0] = torch.nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False).to(device)
    resnet_path = os.path.join(base_dir, "resnet_baseline", "single_task", ds, "checkpoints", "best_model.pth")
    load_weights(resnet_model, resnet_path)
    resnet_model.eval()
    
    # Init ViT
    vit_model = ViTBaseline(task_configs, model_size="tiny", pretrained=False, in_chans=n_channels).to(device)
    vit_path = os.path.join(base_dir, "vit_baseline", "single_task", ds, "checkpoints", "best_model.pth")
    load_weights(vit_model, vit_path)
    vit_model.eval()
    
    y_true, y_mamba, y_resnet, y_vit = [], [], [], []
    loss_mamba_total, loss_resnet_total, loss_vit_total = 0.0, 0.0, 0.0
    samples = 0
    
    print(f"Running inference on {ds} ...")
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y_tensor = y.to(device).squeeze(-1) if y.ndim > 1 else y.to(device)
            
            # Process labels
            y_list = y.squeeze().tolist()
            if type(y_list) == int: y_list = [y_list]
            y_true.extend(y_list)
            
            # Predict
            out_m, _ = mamba_model.forward_single(x, ds)
            out_r, _ = resnet_model.forward_single(x, ds)
            out_v, _ = vit_model.forward_single(x, ds)
            
            loss_mamba_total += loss_fn(out_m, y_tensor).item() * x.size(0)
            loss_resnet_total += loss_fn(out_r, y_tensor).item() * x.size(0)
            loss_vit_total += loss_fn(out_v, y_tensor).item() * x.size(0)
            samples += x.size(0)
            
            y_mamba.extend(out_m.argmax(dim=1).cpu().tolist())
            y_resnet.extend(out_r.argmax(dim=1).cpu().tolist())
            y_vit.extend(out_v.argmax(dim=1).cpu().tolist())
            
    # Get val loss from metrics.csv
    def get_val_loss(path):
        try:
            df = pd.read_csv(path)
            return df[df['split'] == 'val']['total_loss'].min()
        except: return None

    val_loss_m = get_val_loss(os.path.join(base_dir, "medical_mamba", "single_task", "patch_16", ds, 'metrics.csv'))
    val_loss_r = get_val_loss(os.path.join(base_dir, "resnet_baseline", "single_task", ds, 'metrics.csv'))
    val_loss_v = get_val_loss(os.path.join(base_dir, "vit_baseline", "single_task", ds, 'metrics.csv'))
    
    loss_records.append({
        'Dataset': ds, 
        'Mamba Val Loss': val_loss_m, 'Mamba Test Loss': loss_mamba_total/samples,
        'ResNet Val Loss': val_loss_r, 'ResNet Test Loss': loss_resnet_total/samples,
        'ViT Val Loss': val_loss_v, 'ViT Test Loss': loss_vit_total/samples,
    })
    
    plot_confusion_matrix(y_true, y_mamba, f"VMamba-Tiny ({ds})", axes[i, 0], n_classes)
    plot_confusion_matrix(y_true, y_resnet, f"ResNet50 ({ds})", axes[i, 1], n_classes)
    plot_confusion_matrix(y_true, y_vit, f"ViT-Tiny ({ds})", axes[i, 2], n_classes)

plt.tight_layout()
plt.show()

In [ ]:
loss_df = pd.DataFrame(loss_records)
display(loss_df)

## 3. Efficiency Focus: Parameters, FLOPs, Inference Latency, and Time Complexity

Here we conduct an empirical investigation comparing the parameter count, floating-point operations (FLOPs), and average latency for single-batch forward passes.

### Theoretical Time Complexity
- **ResNet50 (CNNs)**: Scaling complexity bounded by $O(L \cdot C_{out} \cdot C_{in} \cdot K^2 \cdot H \cdot W)$. It uses local spatial convolutions. When attempting global context representation, CNNs require deep layering or self-attention that carries an $O(N^2)$ quadratic cost to sequence length $N$ where $N=H \cdot W$.
- **VMamba (SSMs)**: Employs Vision State Space Block (VSSB) via Cross-Scan mechanisms. Time complexity for SSM sequence scanning guarantees computational linearity with respect to token length. Asymptotic complexity $O(N \cdot D)$. Hence, VMamba captures global context and scales efficiently without large quadratic memory/time peaks as resolution increases.

In [ ]:
!pip install thop timm -q


In [ ]:
from thop import profile
from torch.profiler import profile as torch_profiler, record_function, ProfilerActivity

dummy_input = torch.randn(1, 3, 224, 224).to(device)
dummy_task = [("mock", 2)]

try: mamba_eval = build_model(dummy_task, model_size="tiny", patch_size=16, in_chans=3).to(device)
except TypeError: mamba_eval = build_model(dummy_task, model_size="tiny", patch_size=16).to(device)
mamba_eval.eval()

resnet_eval = ResNetBaseline(dummy_task, pretrained=False).to(device)
resnet_eval.eval()

vit_eval = ViTBaseline(dummy_task, model_size="tiny", pretrained=False, in_chans=3).to(device)
vit_eval.eval()

def measure_efficiency(model, model_name):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    class Wrapper(torch.nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return self.m.forward_single(x, "mock")[0]
            
    wrapped = Wrapper(model).to(device)
    try:
        macs, _ = profile(wrapped, inputs=(dummy_input, ), verbose=False)
        flops = macs * 2
    except Exception as e:
        flops = 0
        print(f"Profiling FLOPs failed for {model_name}: {e}\n")
    
    for _ in range(10): _ = wrapped(dummy_input)
    runs = 100
    start_time = time.perf_counter()
    for _ in range(runs): _ = wrapped(dummy_input)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    latency_ms = ((time.perf_counter() - start_time) / runs) * 1000
    
    print(f"--- {model_name} Efficiency ---")
    print(f"Parameters: {params / 1e6:.2f} M")
    if flops > 0: print(f"GFLOPs: {flops / 1e9:.3f} G")
    print(f"Inference Latency ({device}): {latency_ms:.2f} ms\n")

measure_efficiency(mamba_eval, "VMamba-Tiny")
measure_efficiency(resnet_eval, "ResNet50")
measure_efficiency(vit_eval, "ViT-Tiny")
